For Fairness Analysis -- adding more data

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

EXTRA_COLS = [
    'interest_rate',
    'rate_spread',
    'denial_reason_1',
    'tract_minority_population_percent',
    'tract_to_msa_income_percentage'
]

chunks = []
chunk_size = 500_000

reader = pd.read_csv(
    '2024_lar.txt',
    sep='|',
    dtype=str,
    usecols=['action_taken'] + EXTRA_COLS,
    chunksize=chunk_size
)

for i, chunk in enumerate(reader):
    filtered = chunk[chunk['action_taken'].isin(['1', '2', '3'])]
    chunks.append(filtered)
    print(f'Chunk {i+1} done: kept {len(filtered):,} rows')

df_raw = pd.concat(chunks, ignore_index=True)
df_raw['approved'] = df_raw['action_taken'].isin(['1', '2']).astype(int)

print(f'\nTotal after action_taken filter: {len(df_raw):,}')

# Identical sample to cleaning notebook
df_sample, _ = train_test_split(
    df_raw,
    train_size=500_000,
    random_state=42,
    stratify=df_raw['approved']
)
df_sample = df_sample.reset_index(drop=True)

print(f'Sample rows: {len(df_sample):,}')

# Load cleaned file
cleaned = pd.read_csv('hmda_2024_cleaned.csv')
print(f'Cleaned rows: {len(cleaned):,}')

# Preview
print('\nExtra cols preview (first 5 rows):')
print(df_sample[EXTRA_COLS].head())
print('\ninterest_rate non-null in sample:', df_sample['interest_rate'].notna().sum())

Chunk 1 done: kept 380,536 rows
Chunk 2 done: kept 358,052 rows
Chunk 3 done: kept 361,183 rows
Chunk 4 done: kept 387,421 rows
Chunk 5 done: kept 370,531 rows
Chunk 6 done: kept 266,673 rows
Chunk 7 done: kept 394,539 rows
Chunk 8 done: kept 413,552 rows
Chunk 9 done: kept 371,482 rows
Chunk 10 done: kept 470,483 rows
Chunk 11 done: kept 346,192 rows
Chunk 12 done: kept 342,036 rows
Chunk 13 done: kept 252,943 rows
Chunk 14 done: kept 258,053 rows
Chunk 15 done: kept 326,238 rows
Chunk 16 done: kept 363,388 rows
Chunk 17 done: kept 334,897 rows
Chunk 18 done: kept 361,154 rows
Chunk 19 done: kept 376,095 rows
Chunk 20 done: kept 367,719 rows
Chunk 21 done: kept 419,142 rows
Chunk 22 done: kept 311,732 rows
Chunk 23 done: kept 264,457 rows
Chunk 24 done: kept 370,967 rows
Chunk 25 done: kept 192,296 rows

Total after action_taken filter: 8,661,761
Sample rows: 500,000
Cleaned rows: 433,173

Extra cols preview (first 5 rows):
  interest_rate rate_spread denial_reason_1 tract_minority_po

In [6]:
# Matching sample data

# Replicate the exempt institution drop (rows where balloon_payment was NaN) by adding a temporary ID to the sample before cleaning drops

# Re-load cleaned file
cleaned = pd.read_csv('hmda_2024_cleaned.csv')

# Convert extra cols to numeric
df_sample['interest_rate'] = pd.to_numeric(
    df_sample['interest_rate'].replace('Exempt', np.nan), errors='coerce'
)
df_sample['rate_spread'] = pd.to_numeric(
    df_sample['rate_spread'].replace('Exempt', np.nan), errors='coerce'
)
df_sample['tract_minority_population_percent'] = pd.to_numeric(
    df_sample['tract_minority_population_percent'], errors='coerce'
)
df_sample['tract_to_msa_income_percentage'] = pd.to_numeric(
    df_sample['tract_to_msa_income_percentage'], errors='coerce'
)

# denial_reason_1: map numeric codes to labels
# Code 10 = not applicable (approved loans have no denial reason)
denial_map = {
    '1':    'Debt-to-income ratio',
    '2':    'Employment history',
    '3':    'Credit history',
    '4':    'Collateral',
    '5':    'Insufficient cash',
    '6':    'Unverifiable information',
    '7':    'Credit application incomplete',
    '8':    'Mortgage insurance denied',
    '9':    'Other',
    '10':   'Not applicable',
    '1111': 'Exempt'
}
df_sample['denial_reason_1'] = df_sample['denial_reason_1'].map(denial_map)

# Step 2: Re-read the full cleaned pipeline to get a surviving row mask by re-running the same pipeline on just the extra cols alongside the full cleaned CSV to find which 433,173 rows survived

# Load the original cleaning notebook's intermediate data
# by reading the raw file again with ALL columns needed for the drop logic
chunks2 = []
reader2 = pd.read_csv(
    '2024_lar.txt',
    sep='|',
    dtype=str,
    usecols=['action_taken', 'balloon_payment', 'income', 'property_value',
             'loan_term', 'applicant_age', 'state_code', 'conforming_loan_limit'],
    chunksize=500_000
)
for chunk in reader2:
    filtered = chunk[chunk['action_taken'].isin(['1', '2', '3'])]
    chunks2.append(filtered)

df_drop_cols = pd.concat(chunks2, ignore_index=True)
df_drop_cols['approved'] = df_drop_cols['action_taken'].isin(['1', '2']).astype(int)

# Take identical sample
df_drop_sample, _ = train_test_split(
    df_drop_cols,
    train_size=500_000,
    random_state=42,
    stratify=df_drop_cols['approved']
)
df_drop_sample = df_drop_sample.reset_index(drop=True)

# Replicate the two drop conditions from Step 6 of cleaning notebook:
# Drop 1: exempt institutions (balloon_payment == '1111' or NaN after replacement)
df_drop_sample['balloon_payment'] = df_drop_sample['balloon_payment'].replace('1111', np.nan)
exempt_mask = df_drop_sample['balloon_payment'].isna()

# Drop 2: missing key variables
df_drop_sample['income']              = pd.to_numeric(df_drop_sample['income'], errors='coerce')
df_drop_sample['property_value']      = pd.to_numeric(
    df_drop_sample['property_value'].replace('Exempt', np.nan), errors='coerce'
)
df_drop_sample['loan_term']           = pd.to_numeric(
    df_drop_sample['loan_term'].replace('Exempt', np.nan), errors='coerce'
)
df_drop_sample['applicant_age']       = df_drop_sample['applicant_age'].replace('8888', np.nan)
key_cols = ['income', 'property_value', 'loan_term', 'applicant_age',
            'state_code', 'conforming_loan_limit']

surviving_mask = ~exempt_mask
surviving_mask = surviving_mask & df_drop_sample[key_cols].notna().all(axis=1)

print(f'Rows surviving both drops: {surviving_mask.sum():,}')
print(f'Cleaned CSV rows:          {len(cleaned):,}')

Rows surviving both drops: 433,173
Cleaned CSV rows:          433,173


In [7]:
# Extract only the surviving rows from df_sample
df_extras = df_sample[EXTRA_COLS].loc[surviving_mask.values].reset_index(drop=True)

print('Extras shape:', df_extras.shape)
print('Cleaned shape:', cleaned.shape)

# Confirm alignment by checking approval rates match
print('\nApproval rate in cleaned:', round(cleaned['approved'].mean() * 100, 2), '%')

# Attach extra columns to cleaned dataset
df_with_extras = cleaned.copy()
df_with_extras['interest_rate'] = df_extras['interest_rate'].values
df_with_extras['rate_spread'] = df_extras['rate_spread'].values
df_with_extras['denial_reason_1'] = df_extras['denial_reason_1'].values
df_with_extras['tract_minority_population_percent'] = df_extras['tract_minority_population_percent'].values
df_with_extras['tract_to_msa_income_percentage'] = df_extras['tract_to_msa_income_percentage'].values

# Checks
print('\nExtra columns added:')
for col in EXTRA_COLS:
    non_null = df_with_extras[col].notna().sum()
    print(f'  {col}: {non_null:,} non-null ({round(non_null/len(df_with_extras)*100, 2)}%)')

print('\nInterest rate among approved only:')
approved_only = df_with_extras[df_with_extras['approved'] == 1]
print(f'  Non-null: {approved_only["interest_rate"].notna().sum():,}')
print(f'  Mean: {approved_only["interest_rate"].mean():.3f}%')
print(f'  Median: {approved_only["interest_rate"].median():.3f}%')

print('\nDenial reason distribution (denied only):')
denied_only = df_with_extras[df_with_extras['approved'] == 0]
print(denied_only['denial_reason_1'].value_counts())

# Save
df_with_extras.to_csv('hmda_2024_cleaned_with_extras.csv', index=False)
print('\nSaved to hmda_2024_cleaned_with_extras.csv')
print('Shape:', df_with_extras.shape)

Extras shape: (433173, 5)
Cleaned shape: (433173, 32)

Approval rate in cleaned: 77.36 %

Extra columns added:
  interest_rate: 334,369 non-null (77.19%)
  rate_spread: 314,976 non-null (72.71%)
  denial_reason_1: 433,173 non-null (100.0%)
  tract_minority_population_percent: 433,173 non-null (100.0%)
  tract_to_msa_income_percentage: 433,173 non-null (100.0%)

Interest rate among approved only:
  Non-null: 334,369
  Mean: 7.147%
  Median: 6.875%

Denial reason distribution (denied only):
denial_reason_1
Debt-to-income ratio             34827
Credit history                   24164
Collateral                       13443
Credit application incomplete    10605
Other                             7358
Unverifiable information          4237
Insufficient cash                 2448
Employment history                 988
Mortgage insurance denied           15
Exempt                               1
Name: count, dtype: int64

Saved to hmda_2024_cleaned_with_extras.csv
Shape: (433173, 37)
